In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

# ========================
# Загрузка данных
# ========================
df_train = pd.read_csv('/kaggle/input/ozon-train/ml_ozon_ounterfeit_train.csv', index_col=0)
df_test = pd.read_csv('/kaggle/input/ozon-test/ml_ozon_ounterfeit_test.csv', index_col=0)
df_test2 = pd.read_csv('/kaggle/input/ozon-cup-new-test/ml_ozon_ounterfeit_new_test.csv', index_col=0)

# Объединяем тесты
new_df = pd.concat([df_test, df_test2], axis=0)
print(f"Размер объединённого теста: {new_df.shape}")

# ========================
# Определение признаков
# ========================
# Все возможные признаки из трейна (исключая таргет)
BASE_FEATURES = [
    'brand_name', 'description', 'name_rus', 'CommercialTypeName4',
    'rating_1_count', 'rating_2_count', 'rating_3_count', 'rating_4_count', 'rating_5_count',
    'comments_published_count', 'photos_published_count', 'videos_published_count',
    'PriceDiscounted', 'item_time_alive',
    'item_count_fake_returns7', 'item_count_fake_returns30', 'item_count_fake_returns90',
    'item_count_sales7', 'item_count_sales30', 'item_count_sales90',
    'item_count_returns7', 'item_count_returns30', 'item_count_returns90',
    'GmvTotal7', 'GmvTotal30', 'GmvTotal90',
    'ExemplarAcceptedCountTotal7', 'ExemplarAcceptedCountTotal30', 'ExemplarAcceptedCountTotal90',
    'OrderAcceptedCountTotal7', 'OrderAcceptedCountTotal30', 'OrderAcceptedCountTotal90',
    'ExemplarReturnedCountTotal7', 'ExemplarReturnedCountTotal30', 'ExemplarReturnedCountTotal90',
    'ExemplarReturnedValueTotal7', 'ExemplarReturnedValueTotal30', 'ExemplarReturnedValueTotal90',
    'ItemVarietyCount', 'ItemAvailableCount', 'seller_time_alive', 'ItemID', 'SellerID'
]

# Категориальные признаки
categorical_features = ['brand_name', 'CommercialTypeName4']

# Текстовые признаки
text_features = ['description', 'name_rus']

# ========================
# Функция создания новых признаков
# ========================
def create_new_features(df):
    df = df.copy()
    
    # 1. Признаки рейтинга
    rating_cols = ['rating_1_count', 'rating_2_count', 'rating_3_count', 'rating_4_count', 'rating_5_count']
    if all(col in df.columns for col in rating_cols):
        total_ratings = df[rating_cols].sum(axis=1)
        df['avg_rating'] = (df['rating_1_count']*1 + df['rating_2_count']*2 + df['rating_3_count']*3 + 
                           df['rating_4_count']*4 + df['rating_5_count']*5) / total_ratings.replace(0, 1)
        df['rating_ratio_4_5'] = (df['rating_4_count'] + df['rating_5_count']) / total_ratings.replace(0, 1)
        df['rating_ratio_1_2'] = (df['rating_1_count'] + df['rating_2_count']) / total_ratings.replace(0, 1)
    
    # 2. Признаки возвратов
    if all(col in df.columns for col in ['item_count_sales7', 'item_count_returns7']):
        df['return_rate_7'] = df['item_count_returns7'] / df['item_count_sales7'].replace(0, 1)
        df['fake_return_rate_7'] = df['item_count_fake_returns7'] / df['item_count_sales7'].replace(0, 1)
    
    if all(col in df.columns for col in ['item_count_sales30', 'item_count_returns30']):
        df['return_rate_30'] = df['item_count_returns30'] / df['item_count_sales30'].replace(0, 1)
        df['fake_return_rate_30'] = df['item_count_fake_returns30'] / df['item_count_sales30'].replace(0, 1)
    
    # 3. Признаки активности
    if all(col in df.columns for col in ['comments_published_count', 'photos_published_count', 'videos_published_count']):
        df['total_engagement'] = df['comments_published_count'] + df['photos_published_count'] + df['videos_published_count']
    
    # 4. Финансовые признаки
    if all(col in df.columns for col in ['GmvTotal7', 'OrderAcceptedCountTotal7']):
        df['avg_order_value_7'] = df['GmvTotal7'] / df['OrderAcceptedCountTotal7'].replace(0, 1)
    
    if all(col in df.columns for col in ['GmvTotal30', 'OrderAcceptedCountTotal30']):
        df['avg_order_value_30'] = df['GmvTotal30'] / df['OrderAcceptedCountTotal30'].replace(0, 1)
    
    # 5. Признаки временных окон
    if all(col in df.columns for col in ['item_count_sales7', 'item_count_sales30', 'item_count_sales90']):
        df['sales_growth_7_30'] = (df['item_count_sales30'] - df['item_count_sales7']) / df['item_count_sales7'].replace(0, 1)
        df['sales_growth_30_90'] = (df['item_count_sales90'] - df['item_count_sales30']) / df['item_count_sales30'].replace(0, 1)
    
    # 6. Признаки доступности
    if all(col in df.columns for col in ['ItemAvailableCount', 'ItemVarietyCount']):
        df['availability_ratio'] = df['ItemAvailableCount'] / df['ItemVarietyCount'].replace(0, 1)
    
    # 7. Текстовые признаки (длина)
    if 'description' in df.columns:
        df['description_length'] = df['description'].fillna('').apply(len)
    
    if 'name_rus' in df.columns:
        df['name_length'] = df['name_rus'].fillna('').apply(len)
    
    # 8. Временные признаки
    if 'item_time_alive' in df.columns:
        df['item_time_alive_log'] = np.log1p(df['item_time_alive'])
    
    if 'seller_time_alive' in df.columns:
        df['seller_time_alive_log'] = np.log1p(df['seller_time_alive'])
    
    return df

# ========================
# Предобработка: трейн и тест
# ========================
def preprocess_data(df, is_train=False):
    df = df.copy()
    
    # Создаем новые признаки
    df = create_new_features(df)
    
    # Таргет (только для трейна)
    if is_train and 'resolution' in df.columns:
        y = df['resolution']
        df = df.drop('resolution', axis=1)
    else:
        y = None

    # Получаем все доступные признаки после создания новых
    all_available_features = [col for col in df.columns if col not in ['resolution']]
    
    # Категориальные - обрабатываем неизвестные значения
    for col in categorical_features:
        if col in df.columns:
            df[col] = df[col].fillna('missing').astype(str)
            # Для тестовых данных заменяем неизвестные значения на 'unknown'
            if not is_train:
                # Сохраняем уникальные значения из тренировочных данных
                train_unique_values = set()
                if 'df_train_processed' in globals():
                    train_unique_values = set(globals()['df_train_processed'][col].unique())
                
                # Заменяем значения, которых нет в тренировочных данных
                df[col] = df[col].apply(lambda x: x if x in train_unique_values else 'unknown')

    # Текстовые
    for col in text_features:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str)

    # Числовые: заполним NaN нулями (если есть)
    numeric_features = [col for col in all_available_features if col not in categorical_features + text_features]
    for col in numeric_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df, y, all_available_features

# Применяем сначала к тренировочным данным
X_train_raw, y_train_raw, ALL_FEATURES = preprocess_data(df_train, is_train=True)

# Сохраняем обработанные тренировочные данные для reference
globals()['df_train_processed'] = X_train_raw.copy()

# Затем применяем к тестовым данным
new_df_processed, _, _ = preprocess_data(new_df, is_train=False)

print(f"X_train_raw shape: {X_train_raw.shape}")
print(f"new_df_processed shape: {new_df_processed.shape}")
print(f"Всего признаков: {len(ALL_FEATURES)}")

# ========================
# Разделение train/val
# ========================
X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.01,
    stratify=y_train_raw,
    random_state=45
)

# ========================
# Обучение моделей на GPU
# ========================
n_folds = 5
n_models = 4
oof_predictions = np.zeros(len(X_train))
test_predictions_proba = np.zeros(len(new_df_processed))
models = []

for model_idx in range(n_models):
    print(f"\n🧪 Обучение модели {model_idx + 1}/{n_models}")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42 + model_idx)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        print(f"   🔹 Fold {fold + 1}/{n_folds}")

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        # Pool с обработкой текста
        try:
            train_pool = Pool(
                data=X_fold_train,
                label=y_fold_train,
                text_features=text_features,
                cat_features=categorical_features
            )
            val_pool = Pool(
                data=X_fold_val,
                label=y_fold_val,
                text_features=text_features,
                cat_features=categorical_features
            )

            params = {
                'iterations': 700,
                'learning_rate': 0.06,
                'depth': 6,
                'l2_leaf_reg': 2,
                'loss_function': 'Logloss',
                'eval_metric': 'AUC',
                'task_type': 'GPU',
                'devices': '0',
                'random_seed': 42 + model_idx * 10 + fold,
                'verbose': 100,
                'early_stopping_rounds': 50,
                'bootstrap_type': 'Bayesian',
                'bagging_temperature': 0.8
            }

            if text_features:
                params['text_processing'] = {
                    'tokenizers': [
                        {'tokenizer_id': 'Space', 'delimiter': ' ', 'token_types': ['Word', 'Number']}
                    ],
                    'dictionaries': [
                        {'dictionary_id': 'Word', 'gram_order': '1'}
                    ],
                    'feature_processing': {
                        'default': [
                            {'dictionaries_names': ['Word'], 'feature_calcers': ['BoW']}
                        ]
                    }
                }

            model = CatBoostClassifier(**params)
            model.fit(train_pool, eval_set=val_pool, use_best_model=True, plot=False)

            # OOF
            oof_preds = model.predict_proba(X_fold_val)[:, 1]
            oof_predictions[val_idx] += oof_preds / n_models

            # Тест
            test_preds = model.predict_proba(new_df_processed)[:, 1]
            test_predictions_proba += test_preds / (n_models * n_folds)

            models.append(model)

        except Exception as e:
            print(f"❌ Ошибка в фолде {fold}: {e}")
            continue

# ========================
# Создание submission
# ========================
# Бинарные предсказания (по порогу 0.5)
test_predictions_binary = (test_predictions_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    'id': new_df_processed.index,
    'prediction': test_predictions_binary
})

submission.to_csv('submission5.csv', index=False)

print("\n✅ Готово!")
print(f"Сохранено: submission_new_df_gpu.csv")
print(f"Количество строк: {len(submission)}")
print(f"Распределение: \n{submission['prediction'].value_counts()}")
print(f"Средняя вероятность: {test_predictions_proba.mean():.4f}")

Размер объединённого теста: (31391, 43)
X_train_raw shape: (197198, 60)
new_df_processed shape: (31391, 60)
Всего признаков: 60

🧪 Обучение модели 1/4
   🔹 Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8317888	best: 0.8317888 (0)	total: 11.8s	remaining: 2h 18m 3s
100:	test: 0.9650195	best: 0.9650195 (100)	total: 13.8s	remaining: 1m 21s
200:	test: 0.9696263	best: 0.9696263 (200)	total: 15.5s	remaining: 38.4s
300:	test: 0.9734493	best: 0.9734493 (300)	total: 17.1s	remaining: 22.7s
400:	test: 0.9750168	best: 0.9750168 (400)	total: 18.7s	remaining: 14s
500:	test: 0.9762298	best: 0.9762298 (500)	total: 20.4s	remaining: 8.1s
600:	test: 0.9773527	best: 0.9773527 (600)	total: 22s	remaining: 3.63s
699:	test: 0.9779608	best: 0.9779626 (698)	total: 23.6s	remaining: 0us
bestTest = 0.9779626131
bestIteration = 698
Shrink model to first 699 iterations.
   🔹 Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8577564	best: 0.8577564 (0)	total: 18.9ms	remaining: 13.2s
100:	test: 0.9658359	best: 0.9658359 (100)	total: 1.94s	remaining: 11.5s
200:	test: 0.9695773	best: 0.9695773 (200)	total: 3.66s	remaining: 9.08s
300:	test: 0.9715456	best: 0.9715456 (300)	total: 5.33s	remaining: 7.07s
400:	test: 0.9739383	best: 0.9739383 (400)	total: 7s	remaining: 5.22s
500:	test: 0.9749940	best: 0.9750004 (499)	total: 8.67s	remaining: 3.44s
600:	test: 0.9759616	best: 0.9759616 (600)	total: 10.3s	remaining: 1.7s
699:	test: 0.9768305	best: 0.9768305 (699)	total: 12s	remaining: 0us
bestTest = 0.9768305123
bestIteration = 699
   🔹 Fold 3/5


...

0:	test: 0.7955471	best: 0.7955471 (0)	total: 17.5ms	remaining: 12.3s
100:	test: 0.9631366	best: 0.9631366 (100)	total: 1.94s	remaining: 11.5s
200:	test: 0.9672793	best: 0.9672808 (199)	total: 3.63s	remaining: 9.01s
300:	test: 0.9701360	best: 0.9701360 (300)	total: 5.34s	remaining: 7.08s
400:	test: 0.9712890	best: 0.9712890 (400)	total: 7.03s	remaining: 5.24s
500:	test: 0.9726481	best: 0.9726481 (500)	total: 8.72s	remaining: 3.46s
600:	test: 0.9740543	best: 0.9740543 (600)	total: 10.4s	remaining: 1.72s
699:	test: 0.9748325	best: 0.9748325 (699)	total: 12.1s	remaining: 0us
bestTest = 0.974832505
bestIteration = 699
   🔹 Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8415931	best: 0.8415931 (0)	total: 19.2ms	remaining: 13.4s
100:	test: 0.9635303	best: 0.9635303 (100)	total: 1.94s	remaining: 11.5s
200:	test: 0.9676574	best: 0.9676574 (200)	total: 3.66s	remaining: 9.08s
300:	test: 0.9709936	best: 0.9709936 (300)	total: 5.34s	remaining: 7.08s
400:	test: 0.9726571	best: 0.9726579 (396)	total: 7.06s	remaining: 5.26s
500:	test: 0.9738754	best: 0.9738754 (500)	total: 8.74s	remaining: 3.47s
600:	test: 0.9748327	best: 0.9748516 (598)	total: 10.4s	remaining: 1.72s
699:	test: 0.9753866	best: 0.9753866 (699)	total: 12.1s	remaining: 0us
bestTest = 0.97538656
bestIteration = 699
   🔹 Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8441375	best: 0.8441375 (0)	total: 18.8ms	remaining: 13.2s
100:	test: 0.9681204	best: 0.9681204 (100)	total: 1.94s	remaining: 11.5s
200:	test: 0.9724827	best: 0.9724827 (200)	total: 3.65s	remaining: 9.07s
300:	test: 0.9748003	best: 0.9748003 (300)	total: 5.32s	remaining: 7.05s
400:	test: 0.9767354	best: 0.9767354 (400)	total: 7.01s	remaining: 5.23s
500:	test: 0.9777189	best: 0.9777189 (500)	total: 8.66s	remaining: 3.44s
600:	test: 0.9784288	best: 0.9784288 (597)	total: 10.3s	remaining: 1.7s
699:	test: 0.9792378	best: 0.9792455 (697)	total: 11.9s	remaining: 0us
bestTest = 0.9792455137
bestIteration = 697
Shrink model to first 698 iterations.

✅ Готово!
Сохранено: submission_new_df_gpu.csv
Количество строк: 31391
Распределение: 
prediction
0    29289
1     2102
Name: count, dtype: int64
Средняя вероятность: 0.0997


In [2]:
test_predictions_binary = (test_predictions_proba >= 0.45).astype(int)

submission = pd.DataFrame({
    'id': new_df_processed.index,
    'prediction': test_predictions_binary
})

submission.to_csv('submission45.csv', index=False)

print("\n✅ Готово!")
print(f"Сохранено: submission_new_df_gpu.csv")
print(f"Количество строк: {len(submission)}")
print(f"Распределение: \n{submission['prediction'].value_counts()}")
print(f"Средняя вероятность: {test_predictions_proba.mean():.4f}")


✅ Готово!
Сохранено: submission_new_df_gpu.csv
Количество строк: 31391
Распределение: 
prediction
0    29023
1     2368
Name: count, dtype: int64
Средняя вероятность: 0.0997


In [3]:
submission.head(10)

,id,prediction
0,17384,0
1,260316,0
2,10610,0
3,205236,0
4,308655,0
5,28369,0
6,153981,0
7,206982,0
8,26823,0
9,254176,0
